# 타이타닉 데이터 품질 점검 (초보자용)

이 노트북에서는 데이터 분석의 첫 단계인 **데이터 품질 점검**을 연습합니다.

- 결측치(Missing Values): 비어 있는 값
- 중복값(Duplicates): 완전히 똑같은 행
- 이상치(Outliers): 다른 값들에 비해 유난히 크거나 작은 값

각 셀을 위에서부터 `Shift + Enter` 로 실행하세요.

## 0. 라이브러리 불러오기

- `pandas`: 표(데이터프레임) 다루기
- `numpy`: 숫자 계산
- `matplotlib`, `seaborn`: 그래프 그리기

한글 그래프가 깨지지 않도록 폰트도 설정합니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="AppleGothic")

print("준비 완료")

## 1. 데이터 불러오기

`2urvived` 는 파이썬 변수명으로 쓸 수 없어서 `Survived` 로 이름을 바꿔줍니다.

In [ ]:
DATA_PATH = "/Users/remchoi/.cache/kagglehub/datasets/heptapod/titanic/versions/1/train_and_test2.csv"

df = pd.read_csv(DATA_PATH)
df = df.rename(columns={"2urvived": "Survived"})

print("행/열:", df.shape)
df.head()

## 2. 결측치 확인

`isnull()` 은 비어 있으면 `True`, `sum()` 은 `True` 를 1로 세어 줍니다.

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
ratio = (missing / len(df) * 100).round(2)

missing_table = pd.DataFrame({"결측치 개수": missing, "비율(%)": ratio})
missing_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

missing.plot(kind="bar", ax=axes[0], color="salmon")
axes[0].set_title("컬럼별 결측치 개수")
axes[0].set_ylabel("개수")

sns.heatmap(df[missing.index].isnull(), cbar=False, cmap="Reds", ax=axes[1])
axes[1].set_title("결측치 위치")

plt.tight_layout()
plt.show()

## 3. 중복값 확인

`duplicated()` 는 앞에 똑같은 행이 있으면 `True` 를 돌려줍니다.

In [ ]:
dup_count = df.duplicated().sum()
id_dup = df["Passengerid"].duplicated().sum()

print("완전 중복 행 개수:", dup_count)
print("Passengerid 중복 개수:", id_dup)

if dup_count > 0:
    display(df[df.duplicated(keep=False)].sort_values("Passengerid"))

## 4. 이상치 확인 - IQR 방식

1. `Q1`(25%), `Q3`(75%) 를 구합니다.
2. `IQR = Q3 - Q1`
3. `하한 = Q1 - 1.5*IQR`, `상한 = Q3 + 1.5*IQR`
4. 이 범위를 벗어나면 이상치로 봅니다.

In [ ]:
numeric_cols = ["Age", "Fare", "sibsp", "Parch"]

rows = []
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    rows.append([col, q1, q3, iqr, lower, upper, count, round(count / len(df) * 100, 2)])

iqr_table = pd.DataFrame(
    rows,
    columns=["컬럼", "Q1", "Q3", "IQR", "하한", "상한", "이상치 수", "비율(%)"],
)
iqr_table

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(4 * len(numeric_cols), 5))

for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df[col], ax=ax, color="lightseagreen")
    ax.set_title(f"{col} 박스플롯")

plt.tight_layout()
plt.show()

## 5. 이상치 확인 - Z-score 방식

`z = (값 - 평균) / 표준편차` 로, 평균에서 얼마나 떨어져 있는지 봅니다.
보통 `|z| > 3` 이면 이상치로 봅니다.

In [ ]:
rows = []
for col in numeric_cols:
    z = (df[col] - df[col].mean()) / df[col].std()
    count = (z.abs() > 3).sum()
    rows.append([col, round(df[col].mean(), 3), round(df[col].std(), 3), count])

zscore_table = pd.DataFrame(rows, columns=["컬럼", "평균", "표준편차", "이상치 수"])
zscore_table

## 6. 정보가 없는(값이 하나뿐인) 컬럼 찾기

`nunique()` 로 고유값 개수를 셉니다. 값이 1개면 분석에 쓸 정보가 없습니다.

In [ ]:
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]

print("상수 컬럼 개수:", len(constant_cols))
constant_cols

## 정리

| 항목 | 결과 |
|---|---|
| 결측치 | `Embarked` 2개 (0.15%) |
| 중복값 | 완전 중복 0개, ID 중복 0개 |
| 이상치 | `Fare` 가 가장 많음 (IQR 171개, z-score 38개) |
| 상수 컬럼 | `zero`~`zero.18` 19개 → 삭제 권장 |

### 다음 단계
1. `zero.*` 상수 컬럼 제거
2. 이상치 처리 (제거 / 캡핑 / 로그 변환)
3. 결측치 대체 (평균 / 중앙값 / 최빈값)
4. 탐색적 데이터 분석(EDA) 및 시각화